In [1]:
import xarray as xr
import numpy as np
import xesmf as xe
from dask.diagnostics import ProgressBar
import json


In [2]:
with open("./data/bbox.json", "r") as f:
    bbox = json.load(f)

res = 0.125

In [3]:

def regrid_xarray(dataset, bbox, res):    
    min_lon, min_lat, max_lon, max_lat = [bbox["min_lon"], bbox["min_lat"], bbox["max_lon"], bbox["max_lat"]]

    new_lons = np.arange(min_lon, max_lon + res, res)
    new_lats = np.arange(min_lat, max_lat + res, res)

    ds_tgt = xr.Dataset({
        'lat': (['lat'], new_lats),
        'lon': (['lon'], new_lons)})

    regridder = xe.Regridder(dataset, ds_tgt, 'conservative')
    with ProgressBar():
        ds_regridded = regridder(dataset).compute()

    return ds_regridded

In [4]:
cdm = xr.open_zarr("./resources/copernicus_marine_service/cmems_obs-oc_glo_bgc-optics_my_l4-multi-4km_P1M.zarr")
chl = xr.open_zarr("./resources/copernicus_marine_service/cmems_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M.zarr")
pp = xr.open_zarr("./resources/copernicus_marine_service/cmems_obs-oc_glo_bgc-pp_my_l4-multi-4km_P1M.zarr")
transp = xr.open_zarr("./resources/copernicus_marine_service/cmems_obs-oc_glo_bgc-transp_my_l4-multi-4km_P1M.zarr")


In [5]:
cdm_regridded = regrid_xarray(cdm, bbox, res)
chl_regridded = regrid_xarray(chl, bbox, res)
pp_regridded = regrid_xarray(pp, bbox, res)
transp_regridded = regrid_xarray(transp, bbox, res)

[########################################] | 100% Completed | 2.13 ss
[########################################] | 100% Completed | 445.38 ms
[########################################] | 100% Completed | 439.88 ms
[########################################] | 100% Completed | 550.39 ms


In [6]:
cdm_regridded.to_zarr("./resources/copernicus_marine_service/regridded/obs-oc_glo_optics.zarr", mode="w")
chl_regridded.to_zarr("./resources/copernicus_marine_service/regridded/obs-oc_glo_plankton.zarr", mode="w")
pp_regridded.to_zarr("./resources/copernicus_marine_service/regridded/obs-oc_glo_pp.zarr", mode="w")
transp_regridded.to_zarr("./resources/copernicus_marine_service/regridded/obs-oc_glo_transp.zarr", mode="w")